# Inherit the kind through an intermediate class

Shared fields can live between the marker and a concrete entity. Every class in this chain keeps the CoreEntity suffix.

Use the repository environment for this branch. Run cells in order in a fresh kernel; no published-package install is needed.

## Import the types

In [ ]:
!pip install -q aoa-action-machine

In [2]:
from abc import ABC, abstractmethod

from pydantic import Field

from aoa.action_machine.domain import BaseCoreEntity
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.graph.core.exclude_graph_model import exclude_graph_model
from aoa.action_machine.intents.entities import entities
from aoa.action_machine.intents.entity import entity
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.resources.base_storage import BaseStorage

## Declare the model

In [3]:
class ShopDomain(BaseDomain):
    """Group the shop model declarations."""

    name = "shop"
    description = "Shop model"


@exclude_graph_model
class BaseDocumentCoreEntity(BaseCoreEntity, ABC):
    """Share document identity without becoming a graph entity."""

    id: str = Field(description="Document identifier")

    @abstractmethod
    def label(self) -> str:
        """Return the document's human label."""
        raise NotImplementedError


@entity(description="Customer invoice", domain=ShopDomain)
class InvoiceCoreEntity(BaseDocumentCoreEntity):
    """Describe one invoice using the shared document fields."""

    def label(self) -> str:
        """Return the invoice label."""
        return f"Invoice {self.id}"


@meta(description="Stores the shop entities")
@entities(
    InvoiceCoreEntity,
)
class ShopStorage(BaseStorage):
    """Hold the shop entities."""

## Run this case

In [4]:
invoice = InvoiceCoreEntity(id="inv-1")
assert isinstance(invoice, BaseCoreEntity)
print(invoice.label())

Invoice inv-1


The hierarchy supplies the kind even though the concrete class does not name BaseCoreEntity directly. ABC and an abstract method govern construction of this intermediate class; exclude_graph_model explicitly keeps it out of discovery. Merely naming a class BaseSomething does neither.